# Python Async programming


In [1]:
import asyncio
import time

In [2]:
# Load asynchronous processing functions

async def load_proc(id: int, wait_time: int, fail: bool = False):
    # Your asynchronous processing code here
    try:
        print(f"Starting async process {id} with wait time: {time.monotonic()}")
        await asyncio.sleep(wait_time)
        print(f"Finished async process {id} with wait time: {time.monotonic()}")    
    except asyncio.CancelledError as e:
        print(f"Process {id} was cancelled: {e}")
        return f"Process {id} failed"
    if fail:
        raise Exception("Forced failure")
    return f"Process {id} finished"

In [3]:
start_time = time.monotonic()
proc_id01, proc_id02 = await asyncio.gather(load_proc(1, 2), load_proc(2, 3))
stop_time = time.monotonic()

print(f"Total time: {stop_time - start_time}")
print(f"Proc. IDs: {proc_id01}, {proc_id02}")


Starting async process 1 with wait time: 351686.506293333
Starting async process 2 with wait time: 351686.506334166
Finished async process 1 with wait time: 351688.507645583
Finished async process 2 with wait time: 351689.507572166
Total time: 3.002381666970905
Proc. IDs: Process 1 finished, Process 2 finished


In [4]:
# Multiple calls
start_time = time.monotonic()
ids = await asyncio.gather(*(load_proc(id, 2) for id in range(1, 4)))
stop_time = time.monotonic()

print(f"Total time: {stop_time - start_time}")
print(f"Proc. IDs: {ids}")

Starting async process 1 with wait time: 351689.518727041
Starting async process 2 with wait time: 351689.518792916
Starting async process 3 with wait time: 351689.518806125
Finished async process 1 with wait time: 351691.519911166
Finished async process 2 with wait time: 351691.52005725
Finished async process 3 with wait time: 351691.520072208
Total time: 2.001936124986969
Proc. IDs: ['Process 1 finished', 'Process 2 finished', 'Process 3 finished']


In [5]:
# Concurrents tasks example

# Create a task
task = asyncio.create_task(load_proc(1, 1, fail=False))
print(f"[BEFORE await] Task is Done? {task.done()}")
result = await task
print(f"[AFTER await] Task is Done? {task.done()}")
print(f"Result: {result}")


[BEFORE await] Task is Done? False
Starting async process 1 with wait time: 351691.535540583
Finished async process 1 with wait time: 351692.536553958
[AFTER await] Task is Done? True
Result: Process 1 finished


In [6]:
# Fail programatically
task = asyncio.create_task(load_proc(2, 1, fail=True))
print(f"[BEFORE await] Task is Done? {task.done()}")
try:
    result = await task
    print(f"Result: {result}")
except Exception as e:
    print(f"Task failed with exception: {e}")
print(f"[AFTER await] Task is Done? {task.done()}")

[BEFORE await] Task is Done? False
Starting async process 2 with wait time: 351692.54683525
Finished async process 2 with wait time: 351693.548020291
Task failed with exception: Forced failure
[AFTER await] Task is Done? True


In [7]:
# Cancel a task example
tasks =[asyncio.create_task(load_proc(id, 5, fail=False)) for id in range(3)]
statuses = [task.done() for task in tasks]
print(f"[BEFORE cancel] Task is Done?")
print(f"Statuses: {statuses}")
try:
    await asyncio.sleep(1)  # Simulate some work before cancelling the task
    for task in tasks:
        task.cancel(msg="Cancelling task after 1 second.")
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    for result, task in zip(results, tasks):
        print(f"Result: {result}, Task Done: {task.get_name()}")
    # print(f"Results: {results}")
except asyncio.CancelledError:
    print("Task was cancelled")
print(f"[AFTER cancel] Task is Done?")
statuses = [task.done() for task in tasks]
print(f"Statuses: {statuses}")


[BEFORE cancel] Task is Done?
Statuses: [False, False, False]
Starting async process 0 with wait time: 351693.557004291
Starting async process 1 with wait time: 351693.557036916
Starting async process 2 with wait time: 351693.557046
Process 0 was cancelled: Cancelling task after 1 second.
Process 1 was cancelled: Cancelling task after 1 second.
Process 2 was cancelled: Cancelling task after 1 second.
Result: Process 0 failed, Task Done: Task-55
Result: Process 1 failed, Task Done: Task-56
Result: Process 2 failed, Task Done: Task-57
[AFTER cancel] Task is Done?
Statuses: [True, True, True]


In [8]:
# Implement timetouts pattern

tasks = [asyncio.wait_for(asyncio.sleep(wait_time), timeout=2) for wait_time in range(5)]

for task in tasks:
    try:
        print("Starting timeout test")
        await task
        print("Timeout test completed successfully")
    except asyncio.TimeoutError:
        print("Operation timed out")
    
# Call load_proc with timeout
async def load_proc_with_timeout(id, wait_time):
    print(f"Starting load_proc with wait_time={wait_time}")
    try:
        async with asyncio.timeout(2):
            await load_proc(id, wait_time)
        print("Load proc completed successfully")
    except asyncio.TimeoutError:
        print("Load proc timed out")
    
# Example usage of load_proc_with_timeout
print("=" * 10)
for id, wait_time in zip(range(5), range(5)):
    await load_proc_with_timeout(id, wait_time)


Starting timeout test
Timeout test completed successfully
Starting timeout test
Timeout test completed successfully
Starting timeout test
Operation timed out
Starting timeout test
Operation timed out
Starting timeout test
Operation timed out
Starting load_proc with wait_time=0
Starting async process 0 with wait time: 351701.577807916
Finished async process 0 with wait time: 351701.578163708
Load proc completed successfully
Starting load_proc with wait_time=1
Starting async process 1 with wait time: 351701.578212833
Finished async process 1 with wait time: 351702.578608125
Load proc completed successfully
Starting load_proc with wait_time=2
Starting async process 2 with wait time: 351702.5787495
Process 2 was cancelled: 
Load proc completed successfully
Starting load_proc with wait_time=3
Starting async process 3 with wait time: 351704.580095958
Process 3 was cancelled: 
Load proc completed successfully
Starting load_proc with wait_time=4
Starting async process 4 with wait time: 351706.

In [18]:
# Queue example

tasks_queue = asyncio.Queue()

producers = [asyncio.create_task(tasks_queue.put(f"task1-{i}")) for i in range(5)]

async def consumers():
    while True:
        task = await tasks_queue.get()
        if str(task).endswith("-4") is True:
            tasks_queue.task_done()
            break
        print(f"Consumed {task}")
        tasks_queue.task_done()
        
consumer = asyncio.create_task(consumers())

await asyncio.gather(*producers, consumer)

print("All tasks have been processed.")


Consumed task1-0
Consumed task1-1
Consumed task1-2
Consumed task1-3
All tasks have been processed.
